# 🔴 Quiz 3: 진로 상담 RAG 에이전트 완성
**PINE 2기 7회차 — Quiz 3 독립 실행 노트북**  
사용 기술: **Microsoft Agent Framework (MAF) 1.0.0** — `@tool` + TF-IDF RAG

---
## 📋 목표
`data/rag_source.md`에서 관련 내용을 검색해 진로 질문에 답하는  
**완전한 RAG 에이전트**를 처음부터 끝까지 구현하세요.

## ⏱ 제한시간: 15분

## 🎯 성공 기준
- [ ] `load_chunks()` 함수로 rag_source.md 청크 분할
- [ ] TF-IDF 벡터화로 검색 엔진 구성
- [ ] `@tool` 데코레이터로 검색 함수를 에이전트에 등록
- [ ] 3개 이상의 진로 질문에 답변
- [ ] 🌟 보너스: Session + RAG 결합

## 💡 MAF @tool + RAG 핵심 패턴
```python
from agent_framework import tool, Agent

@tool(approval_mode="never_require")   # 자동 실행
def search_knowledge_base(query: str) -> str:
    """docstring이 에이전트에게 도구를 설명합니다!"""
    return "검색 결과"

agent = Agent(
    client=client,
    instructions="search_knowledge_base 도구를 사용하여 답하세요.",
    tools=[search_knowledge_base]   # 등록!
)
# 에이전트가 필요 시 자동으로 도구 호출!
```

In [ ]:
# ⚙️ 환경 설정
import os
import numpy as np
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from agent_framework import Agent, tool
from agent_framework.openai import OpenAIChatClient

load_dotenv()

chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    model=os.environ["AZURE_OPENAI_MODEL"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-08-01-preview"),
)
print("✅ 환경 설정 완료!")

In [ ]:
# ============================================================
# 🔴 Quiz 3 — Step 1: load_chunks() 함수 완성
# ============================================================
# 목표: rag_source.md를 읽어서 '---' 기준으로 청크 분할
# ============================================================

def load_chunks(filepath: str, min_len: int = 80) -> list:
    """
    마크다운 파일을 읽어 '---' 구분자로 섹션을 분할합니다.
    
    Args:
        filepath: 파일 경로
        min_len: 이보다 짧은 청크 제외
    Returns:
        list: 청크 텍스트 리스트
    """
    # TODO: 1. 파일 열기 (encoding="utf-8")
    with open(???, "r", encoding="utf-8") as f:
        text = ???
    
    # TODO: 2. '---'으로 분할 후 각 섹션 strip()
    sections = [s.strip() for s in text.split(???) if s.strip()]
    
    # TODO: 3. min_len 이상인 섹션만 반환
    return [s for s in sections if len(s) > ???]


# 테스트
chunks = load_chunks("data/rag_source.md")
print(f"✅ 총 {len(chunks)}개 청크 생성!")
print(f"\n첫 번째 청크 미리보기 (100자):")
print(chunks[0][:100], "...")

In [ ]:
# ============================================================
# 🔴 Quiz 3 — Step 2: TF-IDF 벡터화 & 검색 함수 완성
# ============================================================

# TF-IDF 벡터라이저 (이미 완성)
vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), max_features=5000)
chunk_vectors = vectorizer.fit_transform(chunks)
print(f"✅ {len(chunks)}개 청크 벡터화 완료!")


def search_top_k(query: str, top_k: int = 3) -> list:
    """
    쿼리와 가장 유사한 청크를 반환합니다.
    
    Args:
        query: 검색할 질문 텍스트
        top_k: 반환할 청크 수
    Returns:
        list: [(청크 텍스트, 유사도 점수), ...]
    """
    # TODO: 1. 쿼리 벡터화
    query_vec = vectorizer.transform([???])
    
    # TODO: 2. 코사인 유사도 계산
    sims = cosine_similarity(???, ???)[0]
    
    # TODO: 3. 내림차순 정렬 후 top_k 인덱스
    top_idx = np.argsort(sims)[::-1][:???]
    
    # TODO: 4. 유사도 > 0인 결과만 (청크, 유사도) 튜플로 반환
    return [(chunks[i], float(sims[i])) for i in top_idx if sims[i] > 0]


# 검색 테스트
test_results = search_top_k("AI 관련 직업")
print(f"\n🔍 검색 테스트 결과 {len(test_results)}개:")
for i, (chunk, score) in enumerate(test_results):
    print(f"\n[{i+1}위] 유사도: {score:.3f}")
    print(chunk[:100], "...")

In [ ]:
# ============================================================
# 🔴 Quiz 3 — Step 3: @tool 데코레이터로 검색 함수 래핑
# ============================================================

@tool(approval_mode="never_require")
def search_knowledge_base(query: str) -> str:
    """진로, AI 직업, 학습 팁, PINE 프로그램 정보를 지식베이스에서 검색합니다.
    
    Args:
        query: 검색할 내용 (한국어)
    Returns:
        관련 정보 텍스트
    """
    # TODO: search_top_k()로 검색 후 결과 포맷팅
    results = search_top_k(???, top_k=3)
    
    if not results:
        return "관련 정보를 찾지 못했습니다."
    
    # TODO: "[참고 자료 N]\n{청크}" 형식으로 조합
    parts = []
    for i, (chunk, score) in enumerate(results):
        parts.append(f"[참고 자료 {i+1}] (관련도: {score:.2f})\n{???[:400]}")
    
    return "\n\n".join(???)


# 도구 직접 테스트
print("🔧 @tool 직접 테스트:")
print(search_knowledge_base("AI 관련 직업")[:300], "...")

In [ ]:
# ============================================================
# 🔴 Quiz 3 — Step 4: RAG 에이전트 생성 & 진로 질문 3개!
# ============================================================

# 1️⃣ RAG 에이전트 생성
rag_agent = Agent(
    client=chat_client,
    name="진로상담AI",
    instructions="""???
    # 반드시 포함:
    # - search_knowledge_base 도구 사용 명시
    # - [출처: 참고 자료 N] 표시 요구
    # - 참고 자료에 없으면 솔직하게 말하기
    """,
    tools=[???]  # search_knowledge_base 등록!
)

print("🎓 진로 상담 RAG 에이전트 시작!")
print("=" * 60)

# 2️⃣ 내가 궁금한 진로 질문 3개 작성 & 실행
my_questions = [
    "???",   # 예: "머신러닝 엔지니어가 되려면 어떤 기술이 필요한가요?"
    "???",   # 예: "PINE 프로그램에서 어떤 결과물을 만들 수 있나요?"
    "???",   # 예: "수학을 잘 못해도 AI 분야에 갈 수 있을까요?"
]

for i, q in enumerate(my_questions, 1):
    print(f"\n{'='*60}")
    print(f"❓ 질문 {i}: {q}")
    answer = await rag_agent.run(???)
    print(f"🤖 답변:\n{answer}")

In [ ]:
# ============================================================
# 🌟 보너스: RAG + Session 결합!
# ============================================================

# 세션 생성
rag_session = await rag_agent.???()

print("🌟 RAG + Session 에이전트")
print("=" * 60)

bonus_talks = [
    "안녕! 나는 고2 소연이야. 장래희망이 AI 연구원이야.",
    "AI 연구원이 되려면 어떤 학과에 가야 해?",   # RAG 검색 발동!
    "내 꿈이 뭐라고 했지?",                       # 메모리 확인!
]

for msg in bonus_talks:
    print(f"\n👤 소연: {msg}")
    r = await rag_agent.run(msg, ???=rag_session)  # session= 전달!
    print(f"🤖 AI: {r}")

In [ ]:
# ✅ 정답 예시 (강사 참고용)

# def load_chunks(filepath, min_len=80):
#     with open(filepath, "r", encoding="utf-8") as f:
#         text = f.read()
#     sections = [s.strip() for s in text.split("---") if s.strip()]
#     return [s for s in sections if len(s) > min_len]

# def search_top_k(query, top_k=3):
#     query_vec = vectorizer.transform([query])
#     sims = cosine_similarity(query_vec, chunk_vectors)[0]
#     top_idx = np.argsort(sims)[::-1][:top_k]
#     return [(chunks[i], float(sims[i])) for i in top_idx if sims[i] > 0]

# @tool(approval_mode="never_require")
# def search_knowledge_base(query: str) -> str:
#     """진로, AI 직업, 학습 팁, PINE 프로그램 정보를 검색합니다."""
#     results = search_top_k(query, top_k=3)
#     if not results: return "관련 정보를 찾지 못했습니다."
#     parts = [f"[참고 자료 {i+1}] (관련도: {s:.2f})\n{c[:400]}" for i, (c, s) in enumerate(results)]
#     return "\n\n".join(parts)

# rag_agent = Agent(
#     client=chat_client, name="진로상담AI",
#     instructions="""당신은 고등학생 진로 상담 AI입니다.
# 질문에 답할 때 반드시 search_knowledge_base 도구를 사용하세요.
# 답변 마지막에 [출처: 참고 자료 N] 형식으로 출처를 표시하세요.
# 참고 자료에 없는 내용은 "확인이 필요합니다"라고 말하세요.""",
#     tools=[search_knowledge_base]
# )
# answer = await rag_agent.run("AI 관련 직업에는 어떤 것들이 있나요?")

# # 보너스: Session 결합
# rag_session = await rag_agent.create_session()
# r = await rag_agent.run("안녕! 나는 소연이야.", session=rag_session)